In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import wandbimport numpy as np

ModuleNotFoundError: No module named 'torch'

In [ ]:
import wandb
wandb.init(project="cifar100-cnn-pytorch", config={
    "learning_rate": 0.001,
    "epochs": 5, # num_epochs from the training cell
    "batch_size": 64, # from trainloader
    "test_batch_size": 1000, # from testloader
    "optimizer": "Adam"
})

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Download and load the training data
trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load the test data
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

In [ ]:
# Define the CNN architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # First convolutional layer (3 input channel, 32 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional layer (32 input channels, 64 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # CIFAR-100 images are 32x32. After two 2x2 pooling layers, size becomes 32/2 -> 16/2 -> 8x8
        # So, input features to the linear layer = 64 channels * 8 * 8 image size
        self.fc1 = nn.Linear(64 * 8 * 8, 128) 
        self.fc2 = nn.Linear(128, 100) # Output layer (100 classes for CIFAR-100)

    def forward(self, x):
        # Apply first conv layer, then ReLU, then pooling
        x = self.pool1(F.relu(self.conv1(x)))
        # Apply second conv layer, then ReLU, then pooling
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Flatten the output from conv layers before passing to linear layers
        x = x.view(-1, 64 * 8 * 8) # -1 infers batch size
        
        # Apply first linear layer, then ReLU
        x = F.relu(self.fc1(x))
        # Apply second linear layer (output layer)
        x = self.fc2(x)
        return x

# Instantiate the model
model = Net()

# Print model structure (optional)
print(model)

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the model
num_epochs = 5
train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        images, labels = data
        
        optimizer.zero_grad() # Zero the parameter gradients
        
        outputs = model(images) # Forward pass
        loss = criterion(outputs, labels) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Optimize
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    
    # Validation for this epoch
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad(): # Disable gradient calculations
        for data in testloader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_acc = 100 * correct / total
    val_accuracies.append(epoch_acc)
    print(f'Accuracy of the network on the test images after epoch {epoch + 1}: {epoch_acc:.2f} %')
    
    # Log metrics to W&B
    wandb.log({"epoch": epoch + 1, "loss": epoch_loss, "accuracy": epoch_acc})

print('Finished Training')

In [ ]:
# Final evaluation of the model
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final accuracy of the network on the 10000 test images: {final_accuracy:.2f} %')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure model is in evaluation mode
model.eval()

# Get a few random samples from the testset
num_samples = 5
# Create a random permutation of indices from the testset
random_indices = np.random.choice(len(testset), num_samples, replace=False)

fig, axes = plt.subplots(1, num_samples, figsize=(15, 3)) # Create a figure with subplots

for i, idx in enumerate(random_indices):
    image, label = testset[idx]
    
    # The image is a PyTorch tensor. We need to move it to CPU if it's on GPU,
    # remove the channel dimension for grayscale, and convert to numpy for plotting.
    # Also, unnormalize the image if normalization was applied during data loading.
    # The normalization for CIFAR-100 is Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
    # Unnormalization: image * std + mean
    mean = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1)
    std = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1)
    unnormalized_image = image.cpu() * std + mean # image is a tensor (C, H, W)
                                                             
    ax = axes[i]
    # For imshow, permute to (H, W, C) and convert to numpy
    ax.imshow(unnormalized_image.permute(1, 2, 0).numpy())
    ax.set_title(f"True: {label}")
    ax.axis('off') # Hide axes ticks

    # Make a prediction
    # We need to add a batch dimension (B) and channel dimension (C) to the image (H, W) -> (B, C, H, W)
    # and move it to the same device as the model if necessary.
    with torch.no_grad():
        # Assuming image is (C, H, W) and model expects (B, C, H, W)
        # Also, ensure the image tensor is on the same device as the model
        image_tensor = image.unsqueeze(0) 
        # If your model is on a GPU, move the tensor to GPU:
        # if next(model.parameters()).is_cuda:
        #    image_tensor = image_tensor.to('cuda')
            
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
        ax.set_xlabel(f"Pred: {predicted.item()}")

plt.tight_layout()
plt.show()

# Log image samples to W&B
for i, idx in enumerate(random_indices):
    image_to_log, true_label = testset[idx] # Get the original image tensor and label
    
    # Ensure image_tensor is prepared as it is for the model input for prediction
    image_tensor_for_pred = image_to_log.unsqueeze(0)
    # if next(model.parameters()).is_cuda: # if model is on GPU
    #    image_tensor_for_pred = image_tensor_for_pred.to('cuda')
    
    with torch.no_grad():
        output = model(image_tensor_for_pred)
        _, predicted_label = torch.max(output.data, 1)
    
    # Log the image, true label, and predicted label to W&B
    wandb.log({f"test_sample_{i}": wandb.Image(image_to_log, caption=f"True: {true_label}, Pred: {predicted_label.item()}")})

# Print predicted and true labels below the images for clarity
print("Details of the samples shown above:")
for i, idx in enumerate(random_indices):
    _, label = testset[idx] # Original label
    
    # Re-predict to match the loop structure for printing
    image_tensor = testset[idx][0].unsqueeze(0)
    # if next(model.parameters()).is_cuda:
    #    image_tensor = image_tensor.to('cuda')
        
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
    
    print(f"Sample {i+1}: True Label = {label}, Predicted Label = {predicted.item()}")


In [ ]:
import matplotlib.pyplot as plt

# Assuming train_losses and val_accuracies are populated from the training loop
# and num_epochs variable is available from the training cell, or derive from list length.
num_epochs_actual = range(1, len(train_losses) + 1)

# Plot 1: Training Loss vs. Epochs
plt.figure()
plt.plot(num_epochs_actual, train_losses, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss over Epochs')
plt.show()

# Plot 2: Validation Accuracy vs. Epochs
plt.figure()
plt.plot(num_epochs_actual, val_accuracies, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Validation Accuracy over Epochs')
plt.show()

In [ ]:
wandb.finish()